In [ ]:
from anomalib.data import Folder
from anomalib.models import Padim, Patchcore
from anomalib.engine import Engine
import torch

def train_socket_inspector():
    """ model = Patchcore(
        backbone="resnet18",  # Better feature extraction
        layers=["layer2", "layer3"],  # Multi-scale features
    )
    
    # More training configs
    datamodule = Folder(
        name="socket_pins",
        root="./datasets/socket_pins", 
        normal_dir="train/good",
    ) """

    model = Padim(backbone="resnet18")
    datamodule = Folder(
        name="socket_pins",
        root="./datasets/socket_pins", 
        normal_dir="train/good",
    )
    
    # 3. Training Engine
    # AUROC score to judge accuracy
    engine = Engine()
    
    # 4. Fit the model
    print("Starting training...")
    engine.fit(datamodule=datamodule, model=model)
    """ engine.test(datamodule=datamodule, model=model) """
    print("Training complete. Model saved.")

    engine.export(
        model=model,
        export_type="torch", # or "openvino" for Intel optimization
        ckpt_path="results/Padim/socket_pins/latest/weights/lightning/model.ckpt",
    )
    print("Model exported to .pt format successfully.")

if __name__ == "__main__":
    train_socket_inspector()

In [ ]:
from anomalib.deploy import TorchInferencer, OpenVINOInferencer
import cv2
import matplotlib.pyplot as plt
import dotenv
import os
dotenv.load_dotenv()

def inspect_pin(model_path, pin_image_path):
    inferencer = TorchInferencer(path=model_path, device="cpu")
    predictions = inferencer.predict(image=pin_image_path)
    
    # --- HANDLE HEATMAP SHAPE ---
    anom_map = predictions.anomaly_map
    if hasattr(anom_map, "cpu"): 
        anom_map = anom_map.cpu().numpy()
    anom_map = anom_map.squeeze() # Removes (1, 256, 256) -> (256, 256)

    # --- HANDLE SCORE FORMATTING ---
    # Convert Tensor to float using .item()
    score = predictions.pred_score.item() 

    # Visualize
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    
    # Original Image
    ax[0].imshow(cv2.cvtColor(cv2.imread(pin_image_path), cv2.COLOR_BGR2RGB))
    ax[0].set_title("Input Pin")
    ax[0].axis('off')
    
    # Anomaly Heatmap
    ax[1].imshow(anom_map, cmap='inferno') 
    ax[1].set_title(f"Score: {score:.2f}") # Now valid because 'score' is a float
    ax[1].axis('off')
    
    plt.tight_layout()
    plt.show()

img_path = "./datasets/socket_pins/test/defect/"
for img in os.listdir(img_path):
    inspect_pin("./results/Padim/socket_pins/latest/weights/torch/model.pt", f"{img_path}{img}")

In [2]:
import fiftyone as fo 
import fiftyone.brain as fob # ML methods
import fiftyone.zoo as foz # zoo datasets and models
from fiftyone import ViewField as F # helper for defining views
import fiftyone.utils.huggingface as fouh # Hugging Face integration

OBJECT="socket_pin"

# Initialize Dataset
if "socket_dataset" in fo.list_datasets():
    fo.delete_dataset("socket_dataset")
dataset = fo.Dataset("socket_dataset")

samples = []
data_path = "./datasets/socket_pins"

train_dir = os.path.join(data_path, "train", "good")
if os.path.exists(train_dir):
    for file in os.listdir(train_dir):
        if file.lower().endswith(('.jpg', '.png')):
            sample = fo.Sample(filepath=os.path.join(train_dir, file))
            
            sample["split"] = "train"
            sample["category"] = fo.Classification(label=OBJECT)
            sample["state"] = fo.Classification(label="good")
            
            samples.append(sample)

test_good_dir = os.path.join(data_path, "test", "good")
if os.path.exists(test_good_dir):
    for file in os.listdir(test_good_dir):
        if file.lower().endswith(('.jpg', '.png')):
            sample = fo.Sample(filepath=os.path.join(test_good_dir, file))
            
            sample["split"] = "test"
            sample["category"] = fo.Classification(label=OBJECT)
            sample["state"] = fo.Classification(label="good")
            
            samples.append(sample)

dataset.add_samples(samples)
print(f"Loaded {len(samples)} samples.")


 100% |███████████████| 2153/2153 [1.3s elapsed, 0s remaining, 1.7K samples/s]         
Loaded 2153 samples.


In [4]:
session = fo.launch_app(dataset)